# FINAL 07 — Multi-Dimensional Ethical-Risk Evidence Synthesis

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Synthesise the final evidence streams into a traceable ethical-risk profile without an overall risk score, model ranking or unsupported causal claim.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [9]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

import json, pandas as pd, numpy as np


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN


In [10]:
# Load evidence produced by the preceding notebooks.
perf=pd.read_csv(FINAL["results"]/"F04_final_tweeteval_test.csv")
sub=pd.read_csv(FINAL["results"]/"F05_RQ1_subgroup_performance.csv")
parity=pd.read_csv(FINAL["results"]/"F05_RQ2_CPD_CRR_five_models.csv")
err=pd.read_csv(FINAL["results"]/"F06_all_error_directions.csv")
conf=pd.read_csv(FINAL["results"]/"F07_confidence_calibration_by_subgroup.csv")
cue=pd.read_csv(FINAL["results"]/"F08_explanation_cue_summary.csv")
agreement=pd.read_csv(FINAL["results"]/"F08_SHAP_LIME_cue_agreement.csv")
sample_manifest=pd.read_csv(FINAL["results"]/"F08_explanation_sample_manifest.csv")
s140=pd.read_csv(FINAL["results"]/"F09_Sentiment140_binary_robustness.csv")
se=pd.read_csv(FINAL["results"]/"F10_SemEval_domain_task_shift.csv")
coverage=pd.read_csv(FINAL["results"]/"F09_F10_external_subgroup_coverage.csv")
external_subgroups=pd.read_csv(FINAL["results"]/"F09_F10_external_subgroup_performance.csv")


In [11]:
def top_error():
    r=err.sort_values("N Errors",ascending=False).iloc[0]
    return f"{r['Error Direction']} = {int(r['N Errors'])} errors ({r['% of All Errors']:.2f}%)"

def top_explanation_cue(method):
    part=cue[(cue["Method"]==method) & (cue["Cue"]!="Other lexical/contextual cue")].copy()
    if part.empty:
        return "no specific coded cue"
    r=part.sort_values(["N Sampled Cases","% of Explanation Sample"],ascending=False).iloc[0]
    return f"{r['Cue']} ({int(r['N Sampled Cases'])}/{len(sample_manifest)} sampled cases; {r['% of Explanation Sample']:.1f}%)"

mean_agreement=agreement.iloc[0]["Mean SHAP-LIME Cue Jaccard"]

profile=pd.DataFrame([
    {
        "Dimension":"Predictive performance / linguistic inclusivity",
        "Primary Evidence":f"TweetEval Macro F1={perf.iloc[0]['Macro F1']:.4f}; subgroup performance reported separately with confidence intervals",
        "Evidence Status":"quantitative held-out evidence",
        "Key Limitation":"small subgroups are exploratory below frozen N=30; linguistic groups are not demographic protected classes"
    },
    {
        "Dimension":"Linguistic-subgroup disparity",
        "Primary Evidence":"CPD/CRR across five model families plus one-vs-rest TPR/FPR gap summaries",
        "Evidence Status":"quantitative descriptive disparity evidence",
        "Key Limitation":"describes outcome disparity across writing styles; does not establish demographic unfairness or causal mechanism"
    },
    {
        "Dimension":"Misclassification / confidence / calibration",
        "Primary Evidence":top_error()+f"; max stable-group HCER={conf[conf['Interpretation Status']=='stable_quantitative']['HCER'].max():.4f}",
        "Evidence Status":"complete error population + reliability indicators",
        "Key Limitation":"HCER/MCE/ECE are measurement indicators, not external ethical-risk standards"
    },
    {
        "Dimension":"Explainability",
        "Primary Evidence":(
            f"SHAP/LIME stratified error sample (N={len(sample_manifest)}); "
            f"top LIME cue: {top_explanation_cue('LIME')}; "
            f"top SHAP cue: {top_explanation_cue('SHAP')}; "
            f"mean specific-cue Jaccard={mean_agreement:.4f}" if pd.notna(mean_agreement)
            else
            f"SHAP/LIME stratified error sample (N={len(sample_manifest)}); "
            f"top LIME cue: {top_explanation_cue('LIME')}; "
            f"top SHAP cue: {top_explanation_cue('SHAP')}; "
            "specific-cue agreement undefined where neither method exposed a coded cue"
        ),
        "Evidence Status":"sampled associative explanation evidence with reproducible multi-label cue coding",
        "Key Limitation":"post-hoc explanations and coded cues do not establish causal error mechanisms; sample cue frequencies are not population prevalence"
    },
    {
        "Dimension":"External robustness / generalisation",
        "Primary Evidence":(
            f"Sentiment140 binary Macro F1={s140.iloc[0]['Macro F1 (negative/positive truth labels)']:.4f} "
            f"(Δ vs matched TweetEval binary={s140.iloc[0]['Macro F1 Delta vs TweetEval Source Binary']:+.4f}); "
            f"emoji-heavy Macro F1={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='emoji-heavy')].iloc[0]['Macro F1']:.4f} "
            f"(N={int(external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='emoji-heavy')].iloc[0]['N'])}; "
            f"gap vs reference={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='emoji-heavy')].iloc[0]['Macro F1 Gap vs Reference']:+.4f}); "
            f"slang-heavy Macro F1={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='slang-heavy')].iloc[0]['Macro F1']:.4f} "
            f"(N={int(external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='slang-heavy')].iloc[0]['N'])}; "
            f"gap vs reference={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='slang-heavy')].iloc[0]['Macro F1 Gap vs Reference']:+.4f}); "
            f"sarcasm-indicated Macro F1={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='sarcasm-indicated')].iloc[0]['Macro F1']:.4f} "
            f"(N={int(external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='sarcasm-indicated')].iloc[0]['N'])}; "
            f"gap vs reference={external_subgroups[(external_subgroups['Dataset']=='Sentiment140') & (external_subgroups['Subgroup']=='sarcasm-indicated')].iloc[0]['Macro F1 Gap vs Reference']:+.4f}); "
            f"SemEval Macro F1={se.iloc[0]['Macro F1']:.4f} "
            f"(Δ vs TweetEval three-class={se.iloc[0]['Macro F1 Delta vs TweetEval Source']:+.4f}); "
            "no focal SemEval subgroup meets the frozen adequacy criterion"
        ),
        "Evidence Status":"external stress evidence",
        "Key Limitation":"Sentiment140 is a binary distant-supervision condition; emoji-heavy support is N=72; SemEval is a domain/task-shift stress condition; subgroup comparisons are descriptive and coverage-qualified"
    },
])
profile.to_csv(FINAL["results"]/"F12_ethical_risk_evidence_profile.csv",index=False)
display(profile)


,Dimension,Primary Evidence,Evidence Status,Key Limitation
0,Predictive performance / linguistic inclusivity,TweetEval Macro F1=0.6053; subgroup performanc...,quantitative held-out evidence,small subgroups are exploratory below frozen N...
1,Linguistic-subgroup disparity,CPD/CRR across five model families plus one-vs...,quantitative descriptive disparity evidence,describes outcome disparity across writing sty...
2,Misclassification / confidence / calibration,negative -> neutral = 1755 errors (37.73%); ma...,complete error population + reliability indica...,"HCER/MCE/ECE are measurement indicators, not e..."
3,Explainability,SHAP/LIME stratified error sample (N=33); top ...,sampled associative explanation evidence with ...,post-hoc explanations and coded cues do not es...
4,External robustness / generalisation,Sentiment140 binary Macro F1=0.5804 (Δ vs matc...,external stress evidence,Sentiment140 is a binary distant-supervision c...


## Final synthesis rule
The thesis must not sum these dimensions into one total score, rank models as an overall ethical winner, or convert linguistic disparity directly into demographic-harm claims. Chapter 5 should report the evidence by dimension; Chapter 6 should state what the combined evidence supports and what remains uncertain.

In [12]:
summary={
    "overall_score_created":False,
    "overall_model_ranking_created":False,
    "causal_XAI_claim_permitted":False,
    "demographic_harm_inference_permitted":False,
    "external_fairness_transfer_claim_requires_adequate_coverage":True,
    "profile_file":"F12_ethical_risk_evidence_profile.csv"
}
save_json(summary,FINAL["manifests"]/"FINAL_SYNTHESIS_GUARDRAILS.json")
print(json.dumps(summary,indent=2))

{
  "overall_score_created": false,
  "overall_model_ranking_created": false,
  "causal_XAI_claim_permitted": false,
  "demographic_harm_inference_permitted": false,
  "external_fairness_transfer_claim_requires_adequate_coverage": true,
  "profile_file": "F12_ethical_risk_evidence_profile.csv"
}
